<a href="https://colab.research.google.com/github/Euugene0/11th-ML/blob/Week_4/Week4_%EB%B3%B5%EC%8A%B5%EA%B3%BC%EC%A0%9C_%EA%B9%80%EC%9C%A0%EC%A7%84.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 4장 복습과제 — GBM과 XGBoost

**범위:** 교재 4.5 GBM, 4.6 XGBoost  
**문항:** 코딩 문제 5개  
**방법:** 데이터 준비 셀을 먼저 실행하고, 각 문제 아래의 빈 코드 셀에 코드를 작성하세요. 문제는 위에서 아래 순서대로 이어집니다.  
**참고:** `xgboost` 패키지가 없는 환경이라면 설치한 다음 실행하세요. 정확도와 조기 중단 시점은 실행 환경에 따라 달라질 수 있습니다.


## 데이터 준비 — 그대로 실행

위스콘신 유방암 데이터의 레이블은 `0=악성`, `1=양성`입니다. 전체 데이터에서 학습용 80%와 테스트용 20%를 분리하고, 학습용을 다시 학습·검증 데이터로 나눕니다.


In [ ]:
import numpy as np
import xgboost as xgb
from xgboost import XGBClassifier
from sklearn.datasets import load_breast_cancer
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split

cancer = load_breast_cancer()
X, y = cancer.data, cancer.target

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=156, stratify=y
)
X_tr, X_val, y_tr, y_val = train_test_split(
    X_train, y_train, test_size=0.1, random_state=156, stratify=y_train
)

print('전체 데이터:', X.shape)
print('학습:', X_tr.shape, '검증:', X_val.shape, '테스트:', X_test.shape)
print('레이블:', dict(enumerate(cancer.target_names)))


전체 데이터: (569, 30)
학습: (409, 30) 검증: (46, 30) 테스트: (114, 30)
레이블: {0: np.str_('malignant'), 1: np.str_('benign')}


## 문제 1. GBM 모델 학습과 정확도 평가

`GradientBoostingClassifier`로 유방암 데이터를 분류하세요.

1. `n_estimators=100`, `learning_rate=0.1`, `max_depth=3`, `random_state=42`로 모델 생성
2. **`X_train`, `y_train`**으로 학습하고 `X_test` 예측
3. 테스트 정확도를 소수점 넷째 자리까지 출력


In [ ]:
gb_clf = GradientBoostingClassifier(
    n_estimators=100,
    learning_rate=0.1,
    max_depth=3,
    random_state=42
)

gb_clf.fit(X_train, y_train)

gb_pred = gb_clf.predict(X_test)

print(f'테스트 정확도: {accuracy_score(y_test, gb_pred):.4f}')

테스트 정확도: 0.9825


## 문제 2. 파이썬 래퍼용 `DMatrix` 만들기

학습·검증·테스트 데이터를 `xgb.DMatrix`로 각각 변환하세요.

1. **`X_tr`, `y_tr`**로 `dtr` 생성
2. **`X_val`, `y_val`**로 `dval` 생성
3. **`X_test`, `y_test`**로 `dtest` 생성
4. `num_row()`를 이용해 각각의 데이터 건수 출력


In [ ]:
dtr = xgb.DMatrix(X_tr, label=y_tr)
dval = xgb.DMatrix(X_val, label=y_val)
dtest = xgb.DMatrix(X_test, label=y_test)

print('학습 데이터 건수:', dtr.num_row())
print('검증 데이터 건수:', dval.num_row())
print('테스트 데이터 건수:', dtest.num_row())

학습 데이터 건수: 409
검증 데이터 건수: 46
테스트 데이터 건수: 114


## 문제 3. 파이썬 래퍼 XGBoost 학습과 조기 중단

문제 2의 `dtr`과 `dval`로 모델을 학습하세요.

1. `params` 딕셔너리에 `max_depth=3`, `eta=0.05`, `objective='binary:logistic'`, `eval_metric='logloss'`, `seed=42` 지정
2. 평가 목록 `[(dtr, 'train'), (dval, 'eval')]`을 만들어 학습·검증 데이터 지정
3. `xgb.train()`에 `num_boost_round=200`, `early_stopping_rounds=20` 적용
4. 학습 완료 후 모델의 `best_iteration`과 `best_score` 출력

학습 로그가 너무 길면 `verbose_eval=False`를 추가해도 됩니다.


In [ ]:
params = {
    'max_depth': 3,
    'eta': 0.05,
    'objective': 'binary:logistic',
    'eval_metric': 'logloss',
    'seed': 42
}

evals = [(dtr, 'train'), (dval, 'eval')]

xgb_model = xgb.train(
    params=params,
    dtrain=dtr,
    num_boost_round=200,
    evals=evals,
    early_stopping_rounds=20,
    verbose_eval=False
)

print('best_iteration:', xgb_model.best_iteration)
print('best_score:', xgb_model.best_score)

best_iteration: 198
best_score: 0.06510487797862405


## 문제 4. 예측 확률을 클래스로 바꾸고 평가하기

문제 3의 파이썬 래퍼 모델로 테스트 데이터를 예측하세요.

1. `predict(dtest)`로 **클래스 1의 예측 확률** 계산
2. 확률이 **0.5보다 크면 1**, 그렇지 않으면 **0**으로 변환
3. 예측 확률 앞 5개와 변환한 클래스 앞 5개 출력
4. `accuracy_score()`로 테스트 정확도를 소수점 넷째 자리까지 출력

확률을 클래스로 변환할 때는 리스트 컴프리헨션이나 `np.where()` 중 편한 방법을 사용하세요.


In [ ]:
pred_probs = xgb_model.predict(dtest)

preds = np.where(pred_probs > 0.5, 1, 0)

print('예측 확률 앞 5개:', pred_probs[:5])
print('예측 클래스 앞 5개:', preds[:5])
print(f'테스트 정확도: {accuracy_score(y_test, preds):.4f}')

예측 확률 앞 5개: [0.00198853 0.9975642  0.9854474  0.9988263  0.9826736 ]
예측 클래스 앞 5개: [0 1 1 1 1]
테스트 정확도: 0.9912


## 문제 5. 사이킷런 래퍼 `XGBClassifier` 적용

이번에는 사이킷런 래퍼로 유방암 데이터를 분류하세요.

1. `n_estimators=200`, `learning_rate=0.05`, `max_depth=3`, `objective='binary:logistic'`, `eval_metric='logloss'`, `random_state=42`로 모델 생성
2. **`X_tr`, `y_tr`**로 학습하고 `X_test`에 대해 `predict()` 실행
3. `predict_proba()`에서 **클래스 1의 예측 확률**만 추출
4. 클래스 1의 예측 확률 앞 5개와 테스트 정확도를 소수점 넷째 자리까지 출력

이번 문제에서는 조기 중단을 적용하지 않습니다.


In [ ]:
xgb_wrapper = XGBClassifier(
    n_estimators=200,
    learning_rate=0.05,
    max_depth=3,
    objective='binary:logistic',
    eval_metric='logloss',
    random_state=42
)

xgb_wrapper.fit(X_tr, y_tr)

preds = xgb_wrapper.predict(X_test)
pred_proba = xgb_wrapper.predict_proba(X_test)[:, 1]

print('클래스 1 예측 확률 앞 5개:', pred_proba[:5])
print(f'테스트 정확도: {accuracy_score(y_test, preds):.4f}')

클래스 1 예측 확률 앞 5개: [0.00198853 0.9975642  0.9854474  0.9988263  0.9826736 ]
테스트 정확도: 0.9912


# **LightGBM**
* Heart Failure Clinical Records(심부전 환자 사망 여부 예측) 데이터셋을 활용합니다.
* 환자의 나이, 빈혈 여부, 당뇨 여부, 박출률, 혈청 크레아티닌, 혈청 나트륨, 고혈압 여부, 흡연 여부 등 여러 임상 정보를 바탕으로 생존(0)인지 사망(1)인지 분류합니다.
* 아래 링크에서 데이터셋을 다운 받아주세요.

https://www.kaggle.com/datasets/andrewmvd/heart-failure-clinical-data

* \## 답안 ## 표시된 부분을 채워서 코드를 완성해주세요.

# **데이터 처리**

# 문제 6. 데이터 로딩 및 확인

아래 코드에서 경로를 설정해 데이터를 불러오고,  클래스(`DEATH_EVENT`) 값 분포를 확인해주세요.

In [ ]:
import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

heart_df = pd.read_csv('./heart_failure_clinical_records_dataset.csv')

print(heart_df['DEATH_EVENT'].value_counts())

DEATH_EVENT
0    203
1     96
Name: count, dtype: int64


In [ ]:
# 데이터셋 살펴보기
heart_df.head(3)

,age,anaemia,creatinine_phosphokinase,diabetes,ejection_fraction,high_blood_pressure,platelets,serum_creatinine,serum_sodium,sex,smoking,time,DEATH_EVENT
0,75.0,0,582,0,20,1,265000.00,1.9,130,1,0,4,1
1,55.0,0,7861,0,38,0,263358.03,1.1,136,1,0,6,1
2,65.0,0,146,0,20,0,162000.00,1.3,129,1,1,7,1


# 문제 7. 피처/레이블 분리 및 데이터 분리

* `DEATH_EVENT` 칼럼을 레이블로, 나머지를 피처로 분리해주세요.
* 전체 데이터 중 80%는 학습용, 20%는 테스트용으로 추출해주세요. (`random_state=156`)
* 위에서 만든 학습용 데이터를 다시 90%는 학습용, 10%는 검증용으로 분리해주세요. (`random_state=156`)

In [ ]:
from sklearn.model_selection import train_test_split

X_features = heart_df.drop('DEATH_EVENT', axis=1)
y_label = heart_df['DEATH_EVENT']

X_train, X_test, y_train, y_test = train_test_split(
    X_features, y_label,
    test_size=0.2,
    random_state=156
)

X_tr, X_val, y_tr, y_val = train_test_split(
    X_train, y_train,
    test_size=0.1,
    random_state=156
)

# **LightGBM 적용**

# 문제 8. LightGBM 모델 학습하기

lightgbm 패키지의 LGBMClassifier를 이용하여 학습을 수행해주세요.
* `n_estimators`: 350
* `learning_rate`: 0.05
* 조기중단(`early stopping`) 라운드: 50
* 평가지표: logloss

**참고**: 교재 코드와 달리, 최신 버전의 LightGBM은 조기 중단을 수행할 때 `fit()`의 `early_stopping_rounds` 파라미터를 직접 사용할 수 없습니다. 대신 `callbacks` 파라미터에 `early_stopping()`함수를 리스트로 전달해야 합니다.

In [ ]:
# 아직 설치되어 있지 않다면 아래 셀을 실행해 설치해 주세요.
!pip install lightgbm

In [ ]:
from lightgbm import LGBMClassifier, early_stopping, log_evaluation

lgbm_wrapper = LGBMClassifier(
    n_estimators=350,
    learning_rate=0.05
)

evals = [(X_tr, y_tr), (X_val, y_val)]

lgbm_wrapper.fit(
    X_tr, y_tr,
    eval_metric='logloss',
    eval_set=evals,
    callbacks=[early_stopping(50), log_evaluation(period=10)]
)

preds = lgbm_wrapper.predict(X_test)
pred_proba = lgbm_wrapper.predict_proba(X_test)[:, 1]

[LightGBM] [Info] Number of positive: 69, number of negative: 146
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.002545 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 276
[LightGBM] [Info] Number of data points in the train set: 215, number of used features: 12
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.320930 -> initscore=-0.749500
[LightGBM] [Info] Start training from score -0.749500
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
Training until validation scores don't improve for 50 rounds
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Wa

# 문제 9. 성능 평가하기
교재에서 사용된 `get_clf_eval` 함수입니다. 함수를 실행하고, 그 아래 코드에 알맞은 파라미터를 넣어 성능 평가를 수행해주세요.

In [ ]:
from sklearn.metrics import (accuracy_score, precision_score,
  recall_score, f1_score, roc_auc_score, confusion_matrix)

def get_clf_eval(y_test, pred=None, pred_proba=None):
  confusion = confusion_matrix(y_test, pred)
  accuracy = accuracy_score(y_test, pred)
  precision = precision_score(y_test, pred)
  recall = recall_score(y_test, pred)
  f1 = f1_score(y_test, pred)
  roc_auc = roc_auc_score(y_test, pred_proba)
  print('오차 행렬')
  print(confusion)
  print('정확도:{:.4f}, 정밀도: {:.4f}, 재현율:{:.4f},\
  F1:{:.4f}, AUC: {:.4f}'.format(accuracy, precision, recall, f1, roc_auc))


get_clf_eval(y_test, preds, pred_proba)

오차 행렬
[[37  2]
 [ 5 16]]
정확도:0.8833, 정밀도: 0.8889, 재현율:0.7619,  F1:0.8205, AUC: 0.9292


# **베이지안 최적화 기반의 HyperOpt를 이용한 하이퍼 파라미터 튜닝**

# **HyperOpt 사용하기**

# 문제 10. HyperOpt 기본 사용법 익히기
아래 조건에 맞게 HyperOpt의 hp 모듈로 검색 공간을 설정하고, search_space에서 x, y 입력 변숫값을 추출하여 `retval = x**2 - 20*y`로 계산된 값을 반환하는 목적 함수를 완성해주세요.
* 입력 변수 x: -10부터 10까지 1 간격
* 입력 변수 y: -15부터 15까지 1 간격

In [ ]:
# 아직 설치되어 있지 않다면 아래 셀을 실행해 설치해 주세요.
!pip install hyperopt

In [ ]:
from hyperopt import hp, STATUS_OK

search_space = {
    'x': hp.quniform('x', -10, 10, 1),
    'y': hp.quniform('y', -15, 15, 1)
}

def objective_func(search_space):
    x = search_space['x']
    y = search_space['y']
    retval = x**2 - 20*y

    return retval

# 문제 11. fmin()으로 최적 입력값 찾기
fmin() 함수를 이용해 위에서 설정한 목적 함수의 반환값이 최솟값이 되는 최적 입력 변숫값을 찾아주세요.
* `fn`: 위에서 설정한 목적 함수
* `sapce`: 위에서 설정한 검색 공산
* `algo`: tpe.suggest
* `max_evals`: 20
* `trials`: Trials() 객체
* `rstate=np.random.default_rng(seed=0)`

In [ ]:
from hyperopt import fmin, tpe, Trials

trial_val = Trials()

best_01 = fmin(
    fn=objective_func,
    space=search_space,
    algo=tpe.suggest,
    max_evals=20,
    trials=trial_val,
    rstate=np.random.default_rng(seed=0)
)

print('best:', best_01)

100%|██████████| 20/20 [00:00<00:00, 620.31trial/s, best loss: -296.0]
best: {'x': np.float64(2.0), 'y': np.float64(15.0)}


# **HyperOpt를 이용한 XGBoost 하이퍼 파라미터 최적화**

여기서부터는 문제 6~9에서 만든 Heart Failure Clinical Records(심부전 환자 사망 여부 예측) 데이터셋(X_train, y_train)을 다시 사용합니다.

# 문제 12. 검색 공간 및 목적 함수 설정하기
아래 조건에 맞게 XGBoost 하이퍼 파라미터 검색 공간을 설정하고, 목적 함수를 완성해주세요.

**검색공간**
* max_depth: 4에서 12까지 1 간격
* min_child_weight: 1에서 5까지 1 간격
* learning_rate: 0.01에서 0.15 사이 정규 분포된 값
* colsample_bytree: 0.5에서 0.9 사이 정규 분포된 값


**목적 함수** (교차 검증 cv=3 기반의 평균 정확도(accuracy)를 반환)
1. 검색 공간에서 목적 함수로 입력되는 모든 값은 실수형이므로, 정수형이어야 하는 하이퍼 파라미터는 형변환을 해주세요.
2. HyperOpt의 목적 함수는 최솟값을 반환하도록 최적화해야 하기 때문에, 정확도처럼 값이 클수록 좋은 성능 지표일 경우 -1을 곱한 뒤 반환해야 합니다.

In [ ]:
# 아직 설치되어 있지 않다면 아래 셀을 실행해 설치해 주세요.
!pip install xgboost

In [ ]:
from typing_extensions import DefaultDict
from sklearn.model_selection import cross_val_score
from xgboost import XGBClassifier
from hyperopt import STATUS_OK

xgb_search_space = {
    'max_depth': hp.quniform('max_depth', 4, 12, 1),
    'min_child_weight': hp.quniform('min_child_weight', 1, 5, 1),
    'learning_rate': hp.uniform('learning_rate', 0.01, 0.15),
    'colsample_bytree': hp.uniform('colsample_bytree', 0.5, 0.9)
}

def objective_func(search_space):
    xgb_clf = XGBClassifier(
        n_estimators=100,
        max_depth=int(search_space['max_depth']),
        min_child_weight=int(search_space['min_child_weight']),
        learning_rate=search_space['learning_rate'],
        colsample_bytree=search_space['colsample_bytree'],
        eval_metric='logloss'
    )

    accuracy = cross_val_score(
        xgb_clf,
        X_train,
        y_train,
        scoring='accuracy',
        cv=3
    )

    return {'loss': -1 * np.mean(accuracy), 'status': STATUS_OK}

# 문제 13. fmin()으로 최적 하이퍼 파라미터 도출 및 재학습

1. `max_eval=50`, `rstate=np.random.default_rng(seed=9)`으로 `fmin()`을 호출해 최적 하이퍼 파라미터를 찾아주세요.
2. 도출된 최적 하이퍼 파라미터로 XGBClassifier를 재학습해주세요.
- `n_estimators`:350
- 조기 중단 라운드: 50
- 평가지표: logloss
- 검증 데이터: (X_tr, y_tr), (X_val, y_val)
- 정수형/실수형 하이퍼 파라미터의 형변환에 유의하세요.
3. 문제 3에서 만든 `get_clf_eval()` 함수로 테스트 데이터 세트에 대한 성능을 평가해주세요.


**참고**: 교재 코드와 달리, 최신 버전의 XGBoost에서는 `fit()`의 `eval_metric`, `early_stopping_rounds`, `callbacks` 파라미터를 직접 사용할 수 없습니다. 대신 해당 파라미터들을 `XGBClassifier()` 생성 시 전달해야 하며, `fit()`에는 `eval_set`과 `verbose` 등을 전달합니다.

In [ ]:
from hyperopt import fmin, tpe, Trials

trial_val = Trials()

best = fmin(
    fn=objective_func,
    space=xgb_search_space,
    algo=tpe.suggest,
    max_evals=50,
    trials=trial_val,
    rstate=np.random.default_rng(seed=9)
)

print('best:', best)

100%|██████████| 50/50 [00:09<00:00,  5.37trial/s, best loss: -0.8410864978902954]
best: {'colsample_bytree': np.float64(0.5260010395565794), 'learning_rate': np.float64(0.07025261754267591), 'max_depth': np.float64(6.0), 'min_child_weight': np.float64(4.0)}


In [ ]:
xgb_wrapper = XGBClassifier(
    n_estimators=350,
    learning_rate=round(float(best['learning_rate']), 5),
    max_depth=int(best['max_depth']),
    min_child_weight=int(best['min_child_weight']),
    colsample_bytree=round(float(best['colsample_bytree']), 5),
    eval_metric='logloss',
    early_stopping_rounds=50
)

evals = [(X_tr, y_tr), (X_val, y_val)]

xgb_wrapper.fit(
    X_tr,
    y_tr,
    eval_set=evals,
    verbose=True
)

preds = xgb_wrapper.predict(X_test)
pred_proba = xgb_wrapper.predict_proba(X_test)[:, 1]

[0]	validation_0-logloss:0.59640	validation_1-logloss:0.54849
[1]	validation_0-logloss:0.58842	validation_1-logloss:0.53778
[2]	validation_0-logloss:0.56257	validation_1-logloss:0.51611
[3]	validation_0-logloss:0.54164	validation_1-logloss:0.49834
[4]	validation_0-logloss:0.52848	validation_1-logloss:0.48804
[5]	validation_0-logloss:0.50929	validation_1-logloss:0.47242
[6]	validation_0-logloss:0.49407	validation_1-logloss:0.45927
[7]	validation_0-logloss:0.47812	validation_1-logloss:0.44153
[8]	validation_0-logloss:0.46223	validation_1-logloss:0.43239
[9]	validation_0-logloss:0.44925	validation_1-logloss:0.41974
[10]	validation_0-logloss:0.43752	validation_1-logloss:0.41074
[11]	validation_0-logloss:0.42588	validation_1-logloss:0.40230
[12]	validation_0-logloss:0.41615	validation_1-logloss:0.39309
[13]	validation_0-logloss:0.40972	validation_1-logloss:0.38697
[14]	validation_0-logloss:0.40571	validation_1-logloss:0.38211
[15]	validation_0-logloss:0.40203	validation_1-logloss:0.38393
[1

In [ ]:
get_clf_eval(y_test, preds, pred_proba)

오차 행렬
[[36  3]
 [ 9 12]]
정확도:0.8000, 정밀도: 0.8000, 재현율:0.5714,  F1:0.6667, AUC: 0.8877


# 문제 14. 질문에 대한 답안을 작성해주세요

**1. LightGBM은 일반 GBM 계열의 트리 분할 방법과 다르게 어떤 방식을 사용하나요? 그 방식의 장단점을 간략히 서술하세요.**

LightGBM은 일반적인 GBM이 사용하는 균형 트리 분할 방식인 Level-wise 방식과 달리 Leaf-wise 방식을 사용한다. Leaf-wise 방식은 현재 존재하는 리프 중 손실을 가장 많이 줄일 수 있는 리프를 선택하여 계속 분할한다. 따라서 동일한 분할 횟수에서 더 빠르게 손실을 줄이고 높은 예측 성능을 얻을 수 있다는 장점이 있다. 반면 특정 리프가 계속 깊어져 비대칭적인 트리가 만들어질 수 있으며, 데이터의 양이 적을 경우 과적합이 발생하기 쉽다는 단점이 있다.여기에 입력해주세요.

**2. 베이지안 최적화가 Grid Search 방식에 비해 갖는 장점을 대체 모델과 획득 함수의 역할을 포함하여 서술하세요.**

베이지안 최적화는 Grid Search처럼 모든 하이퍼 파라미터 조합을 일일이 탐색하지 않고, 이전 탐색 결과를 이용하여 다음에 탐색할 값을 결정한다. 이때 대체 모델은 실제 목적 함수의 형태와 성능을 추정하고, 획득 함수는 대체 모델의 정보를 바탕으로 좋은 결과가 기대되는 영역과 아직 충분히 탐색되지 않은 영역을 고려하여 다음 탐색 지점을 선택한다. 따라서 제한된 탐색 횟수에서도 성능이 좋은 하이퍼 파라미터를 효율적으로 찾을 수 있으며, 학습 비용이 큰 모델에서 Grid Search보다 시간과 계산 비용을 줄일 수 있다는 장점이 있다.

# 분류 실습과 스태킹 앙상블

**범위:** 교재 4.5 분류 실습, 4.6 스태킹 앙상블  
**문항:** 코딩 문제 5개  
**준비:** 캐글 신용카드 사기 검출 실습 코드를 띄워놓고 문제를 풀어주세요.  



## 문제 15. GradientBoostingClassifier을 이용한 학습

사이킷런의 GradientBoostingClassifier를 이용하여 위스콘신 유방암 데이터를 분류할 때, 모델 생성 시 n_estimators=100, learning_rate=0.1, max_depth=3, random_state=42로 설정하였다. 이때 X_train, y_train으로 모델을 학습한 후 X_test에 대한 예측을 수행하고 테스트 정확도를 소수점 넷째 자리까지 출력하기 위해 필요한 코드를 작성하세요.

In [ ]:
gb_clf = GradientBoostingClassifier(
    n_estimators=100,
    learning_rate=0.1,
    max_depth=3,
    random_state=42
)

gb_clf.fit(X_train, y_train)

gb_pred = gb_clf.predict(X_test)

print(f'테스트 정확도: {accuracy_score(y_test, gb_pred):.4f}')


테스트 정확도: 0.8500


## 문제 16. 캐글 신용카드 사기 검출 실습을 통한 불균형 데이터 조정

캐글 신용카드 사기 검출(Credit Card Fraud Detection) 실습에서 레이블이 극도로 불균형한 분포를 가질 때, 소수 클래스(이상 데이터)를 증식시키기 위해 원본 데이터의 피처 값들을 변경하여 새로운 데이터를 생성하는 대표적인 오버 샘플링 기법의 명칭은?

1. 언더 샘플링(Under-Sampling)
2. SMOTE
3. IQR 방식
4. StandScaler


정답 : 2. SMOTE

## 문제 17. IQR 방식을 구현한 이상치 제거

신용카드 사기 데이터에서 사기(Fraud, Class=1)와 정량적 상관관계가 높은 특정 피처(예: V14)의 이상치를 IQR 방식을 통해 제거하는 함수 get_outlier()의 일부이다. 사분위수(Percentile)를 이용해 IQR을 구하고 최솟값과 최댓값을 설정하는 빈칸에 들어갈 올바른 코드는?


In [ ]:
import numpy as np

def get_outlier(df=None, column=None, weight=1.5):
    fraud = df[df['Class']==1][column]

    quantile_25 = np.percentile(fraud.values, 25)
    quantile_75 = np.percentile(fraud.values, 75)

    iqr = quantile_75 - quantile_25
    iqr_weight = iqr * weight

    lowest_val = quantile_25 - iqr_weight
    hightest_val = quantile_75 + iqr_weight

    outlier_index = fraud[
        (fraud < lowest_val) | (fraud > hightest_val)
    ].index

    return outlier_index

## 문제 18. 스태킹 앙상블 모델 학습 중 과적합 문제 개선

스태킹 앙상블(Stacking Ensemble) 모델을 학습할 때, 기본 스태킹 모델이 가지는 과적합(Overfitting) 문제를 개선하기 위해 최종 메타 모델을 위한 학습용/테스트용 데이터 세트를 생성할 때 활용하는 핵심 기법은?
1. 그리드 탐색 (Grid Search CV)
2. 교차 검증 기반의 스태킹 (CV 세트 기반의 스태킹)
3. 사분위수 편차 방식 (IQR)
4. K최근접 이웃 보간법 (KNN Imputation)


정답 : 2. 교차 검증 기반의 스태킹(CV 세트 기반의 스태킹)

## 문제 19. 스태킹 앙상블 CV 기반 예측 세트 생성

기본 스태킹 모델이 가지는 과적합(Overfitting) 문제를 해결하기 위해, 교차 검증(Cross Validation) 기반으로 개별 모델의 예측 결과 데이터 세트를 생성하여 최종 메타 모델의 학습용/테스트용 피처 데이터 세트를 만드는 과정에 대한 설명으로 가장 옳은 것은?
1. 개별 모델들이 원본 학습 데이터 전체로 학습한 뒤 테스트 데이터를 예측한 값만 스태킹하여 메타 모델의 학습 피처로 사용한다.
2. 학습 세트를 여러 개의 폴드(Fold)로 분리한 뒤, 검증 폴드로 예측한 결과값들을 쌓아서 최종 메타 모델의 학습 데이터로 사용하고, 테스트 세트의 예측값들은 최종 평균낸 값을 메타 모델의 테스트 데이터로 사용한다.
3. 스태킹 앙상블을 수행할 때는 개별 모델의 하이퍼파라미터 튜닝을 전혀 진행하지 않고 기본값으로만 결합해야만 성능이 보장된다.
4. 메타 모델은 반드시 로지스틱 회귀 모델만 사용해야 하며, 트리 기반 알고리즘은 메타 모델로 사용할 수 없다.




```
# 코드로 형식 지정됨
```

정답 : 2번